## Table 1. Characteristics of all studies included in the index and validation datasets.

In [2]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "NMWI")
os.chdir(project_root)

In [3]:
file_path = 'Data/Input/Additional file 1.xlsx'
meta = pd.read_excel(file_path, sheet_name='Table S1', skiprows=1)
meta["Citation"] = meta["Author"].astype(str) + " et al., " + meta["Publication Year"].astype(str)

sra_data_full = pd.read_excel(file_path, sheet_name='Table S2', skiprows=1)
platform = meta[['BioProject Number', 'Sequencing Platform']].drop_duplicates()

sra_data_full = sra_data_full.merge(platform, on='BioProject Number', how='left', validate='many_to_one')

In [4]:
df = sra_data_full.copy()

def fmt_num(x):
    return f"{x:g}"

def age_summary(x):
    x = x.dropna()
    return f"{fmt_num(x.median())} [{fmt_num(x.min())}–{fmt_num(x.max())}]" if len(x) else "NA"

def unique_vals(x):
    return ", ".join(map(str, x.dropna().unique())) or "NA"

def female_summary(x):
    x = x.dropna().astype(str).str.lower()
    n_female = x.isin(['f', 'female']).sum()
    return f"{n_female} ({n_female / len(x) * 100:.1f}%)" if len(x) else "NA"

# Sample counts: training = n, validation = (n), both = n(n)
sample_counts = df.groupby(['BioProject Number', 'Set'])['Sample ID'].nunique().unstack(fill_value=0)
sample_counts['Samples'] = sample_counts.apply(
    lambda x: f"{x.get('Training', 0)}({x.get('Validation', 0)})" if x.get('Training', 0) and x.get('Validation', 0)
    else f"{x.get('Training', 0)}" if x.get('Training', 0)
    else f"({x.get('Validation', 0)})", axis=1
)

# One row per study
table1 = df.groupby('BioProject Number').agg(
    Author=('Author', 'first'),
    Year=('Publication Year', 'first'),
    Phenotype=('Disease', unique_vals),
    Age=('Age', age_summary),
    Female=('Sex', female_summary),
    Specimen=('Collection Technique', unique_vals),
    Site=('Sampling Location', unique_vals),
    Country=('Country', unique_vals),
    Region=('16S Region', unique_vals),
    Sequencing_Platform=('Sequencing Platform', unique_vals)
).reset_index()

table1 = table1.merge(sample_counts[['Samples']], on='BioProject Number', how='left')
table1.insert(0, 'Citation', table1['Author'] + ' et al., ' + table1['Year'].astype(str))

table1 = table1[[
    'Citation', 'Phenotype', 'Samples', 'Age', 'Female', 'Specimen',
    'Sequencing_Platform', 'Site', 'Country', 'Region', 'BioProject Number'
]]

table1.columns = [
    'Citation',
    'Clinical phenotype(s)',
    'Samples in training (validation)',
    'Median age, years [range]',
    'Female, n (%)',
    'Specimen type',
    'Sequencing_Platform',
    'Anatomical site (sampling location)',
    'Country of origin',
    '16S rRNA gene region',
    'NCBI BioProject accession'
]

table1 = table1.sort_values('Citation').reset_index(drop=True)
table1

,Citation,Clinical phenotype(s),Samples in training (validation),"Median age, years [range]","Female, n (%)",Specimen type,Sequencing_Platform,Anatomical site (sampling location),Country of origin,16S rRNA gene region,NCBI BioProject accession
0,"Armbruster et al., 2024",CF,16,NA,NA,Mucosal swab,Illumina NovaSeq 6000,Maxillary sinus,USA,V4,PRJNA1081394
1,"Bartosik et al., 2023","Healthy, CRS",31,31 [21–67],11 (35.5%),Mucosal swab,Illumina MiSeq,Middle meatus,Austria,V3–V4,PRJNA907202
2,"Bin-Joo et al., 2024","Healthy, RA",28,NA,NA,Mucosal swab,Illumina MiSeq,Middle meatus,South Korea,V3–V4,PRJNA1058141
3,"Biswas et al., 2017",CRS,13,NA,NA,Mucosal swab,Illumina MiSeq,Middle meatus,New Zealand,V3–V4,PRJNA390854
4,"Chen et al., 2022","Asthma, AR",87,37 [20–67],47 (54.0%),Nasal lavage fluid,Illumina HiSeq 2500,Nasal cavity (unspecified),China,V3–V4,PRJNA793600
5,"Cleary et al., 2021",Healthy,12,27.5 [19–51],10 (83.3%),Mucosal swab,Illumina NextSeq 500,Anterior nares,Malaysia,V4,PRJEB38610
6,"Cope et al., 2017",CRS,17,37 [18–62],NA,Mucosal brushing,Illumina MiSeq,Sinus (unspecified),USA,V4,PRJEB20232
7,"De Boeck et al., 2019","Healthy, CRS, CRS-AS",185,45 [18–65],37 (31.9%),Mucosal swab,Illumina MiSeq,Anterior nares,Belgium,V4,PRJEB30316
8,"Durack et al., 2018","Healthy, Asthma",28,NA,NA,Mucosal brushing,Illumina MiSeq,Nasal cavity (unspecified),USA,V4,PRJEB22676
9,"Fernandez-Rodriguez et al., 2024",Healthy,(19),NA,NA,Mucosal swab,Illumina MiSeq,Anterior nares,USA,V1–V2,PRJNA1024520


In [5]:
# Table 2
df['Health status'] = np.where(
    df['Disease'].astype('string').str.strip().str.lower().eq('healthy'),
    'Healthy', 'Non-healthy'
)
df.loc[df['Disease'].isna(), 'Health status'] = pd.NA

for col in ['Age']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['Sex', 'Sampling Location', '16S Region', 'Sequencing Platform']:
    df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)

d = df[
    df['Set'].isin(['Training', 'Validation']) &
    df['Health status'].notna()
].drop_duplicates(['BioProject Number', 'Sample ID', 'Set', 'Health status']).copy()

groups = [('Training', 'Healthy'), ('Training', 'Non-healthy'),
          ('Validation', 'Healthy'), ('Validation', 'Non-healthy')]

def g(s, h):
    return d[(d['Set'] == s) & (d['Health status'] == h)]

def n_pct(n, total):
    p = 100 * n / total if total else 0
    return f"{n} ({p:.1f}%)"

def study_info(x, var):
    return n_pct(
        x.loc[x[var].notna(), 'BioProject Number'].nunique(),
        x['BioProject Number'].nunique()
    )

def sample_info(x, var):
    return n_pct(x[var].notna().sum(), x['Sample ID'].nunique())

def cat_summary(var, level):
    out = []
    for s, h in groups:
        x = g(s, h)[var].dropna().astype(str).str.strip()
        out.append(n_pct((x == level).sum(), len(x)))
    return out

def male_summary(x):
    x = x.dropna().astype(str).str.lower()
    n_male = x.isin(['m', 'male']).sum()
    return f"{n_male} ({n_male / len(x) * 100:.1f}%)" if len(x) else "NA"

rows = []
def add(label, values):
    rows.append([label, *values])

add('Number of studies', [g(s, h)['BioProject Number'].nunique() for s, h in groups])
add('Number of samples', [g(s, h)['Sample ID'].nunique() for s, h in groups])

add('Age', [''] * 4)
add('  Studies with information, n (%)', [study_info(g(s, h), 'Age') for s, h in groups])
add('  Samples with information, n (%)', [sample_info(g(s, h), 'Age') for s, h in groups])
add('  Median [range]', [age_summary(g(s, h)['Age']) for s, h in groups])

add('Sex', [''] * 4)
add('  Studies with information, n (%)', [study_info(g(s, h), 'Sex') for s, h in groups])
add('  Samples with information, n (%)', [sample_info(g(s, h), 'Sex') for s, h in groups])
add('  Female, n (%)', [female_summary(g(s, h)['Sex']) for s, h in groups])
add('  Male, n (%)', [male_summary(g(s, h)['Sex']) for s, h in groups])


sections = {
    'Sampling location': 'Sampling Location',
    'Hypervariable region': '16S Region',
    'Sequencing Platform': 'Sequencing Platform'
}

for section, var in sections.items():
    levels = d[var].dropna().astype(str).str.strip().drop_duplicates().tolist()
    add(section, [''] * 4)
    add('  Samples with information, n (%)', [sample_info(g(s, h), var) for s, h in groups])
    for level in levels:
        add(f'  {level}, n (%)', cat_summary(var, level))

table2 = pd.DataFrame(rows, columns=[
    'Characteristic',
    'Training | Healthy', 'Training | Non-healthy',
    'Validation | Healthy', 'Validation | Non-healthy'
]).set_index('Characteristic')

table2.columns = pd.MultiIndex.from_tuples([
    ('Training', 'Healthy'), ('Training', 'Non-healthy'),
    ('Validation', 'Healthy'), ('Validation', 'Non-healthy')
])

table2

Training                 \
                                          Healthy    Non-healthy   
Characteristic                                                     
Number of studies                              17             23   
Number of samples                             589           1065   
Age                                                                
  Studies with information, n (%)       7 (41.2%)     12 (52.2%)   
  Samples with information, n (%)     293 (49.7%)    736 (69.1%)   
  Median [range]                       37 [18–84]     42 [18–83]   
Sex                                                                
  Studies with information, n (%)       7 (41.2%)     10 (43.5%)   
  Samples with information, n (%)     293 (49.7%)    708 (66.5%)   
  Female, n (%)                       180 (61.4%)    301 (42.5%)   
  Male, n (%)                         113 (38.6%)    407 (57.5%)   
Sampling location                                                  
  Samples with information, n (%)    589 (100.0%)  1065 (100.0%)   
  Middle meatus, n (%)                245 (41.6%)    555 (52.1%)   
  Anterior nares, n (%)               221 (37.5%)    161 (15.1%)   
  Nasal cavity (unspecified), n (%)     46 (7.8%)    145 (13.6%)   
  Inferior turbinate, n (%)             25 (4.2%)      52 (4.9%)   
  Olfactory cleft, n (%)                52 (8.8%)       0 (0.0%)   
  Maxillary sinus, n (%)                 0 (0.0%)      88 (8.3%)   
  Sinus (unspecified), n (%)             0 (0.0%)      17 (1.6%)   
  Frontal sinus, n (%)                   0 (0.0%)      47 (4.4%)   
Hypervariable region                                               
  Samples with information, n (%)    589 (100.0%)  1065 (100.0%)   
  V3–V4, n (%)                        262 (44.5%)    758 (71.2%)   
  V4, n (%)                           164 (27.8%)    237 (22.3%)   
  V1–V2, n (%)                          41 (7.0%)      56 (5.3%)   
  V1–V3, n (%)                        122 (20.7%)       0 (0.0%)   
  V5–V6, n (%)                           0 (0.0%)       0 (0.0%)   
  V1–V4, n (%)                           0 (0.0%)      14 (1.3%)   
Sequencing Platform                                                
  Samples with information, n (%)    589 (100.0%)  1065 (100.0%)   
  Illumina MiSeq, n (%)               534 (90.7%)    896 (84.1%)   
  Illumina NextSeq 500, n (%)           12 (2.0%)       0 (0.0%)   
  Illumina NovaSeq 6000, n (%)          25 (4.2%)      68 (6.4%)   
  Illumina HiSeq 2500, n (%)            18 (3.1%)     101 (9.5%)   

                                       Validation                
                                          Healthy   Non-healthy  
Characteristic                                                   
Number of studies                               6             8  
Number of samples                             109           241  
Age                                                              
  Studies with information, n (%)       2 (33.3%)     3 (37.5%)  
  Samples with information, n (%)      39 (35.8%)    63 (26.1%)  
  Median [range]                       35 [20–77]    52 [24–99]  
Sex                                                              
  Studies with information, n (%)       2 (33.3%)     3 (37.5%)  
  Samples with information, n (%)      39 (35.8%)    63 (26.1%)  
  Female, n (%)                        21 (53.8%)    21 (33.3%)  
  Male, n (%)                          18 (46.2%)    42 (66.7%)  
Sampling location                                                
  Samples with information, n (%)    109 (100.0%)  241 (100.0%)  
  Middle meatus, n (%)                 18 (16.5%)      0 (0.0%)  
  Anterior nares, n (%)                41 (37.6%)    27 (11.2%)  
  Nasal cavity (unspecified), n (%)    32 (29.4%)    97 (40.2%)  
  Inferior turbinate, n (%)            18 (16.5%)    69 (28.6%)  
  Olfactory cleft, n (%)                 0 (0.0%)      0 (0.0%)  
  Maxillary sinus, n (%)                 0 (0.0%)    48 (19.9%)  
  Si